# 🏠 House Price Prediction — V13 (Triple Ensemble + Stacking + Deep Features)
**Các cải tiến so với V11**:
- **CatBoost** thay thế LightGBM trong ensemble hoặc thêm vào triple ensemble
- **Stacking** với meta-model thay vì blending thủ công
- **TF-IDF + SVD** trích xuất 20 chiều latent từ mô tả
- **Ward-level statistics** (PPM median, price volatility)
- **GroupKFold CV** theo location cluster để tránh data leakage
- **Isolation Forest** cho outlier detection tinh vi
- **Optuna với nhiều tham số hơn** (gamma, reg_alpha, reg_lambda, lambda_l1, lambda_l2)
- **SHAP feature selection** tự động loại bỏ features không quan trọng

## 1. Install packages

In [ ]:
!pip install xgboost lightgbm catboost category_encoders underthesea optuna google-cloud-bigquery scikit-learn shap -qq

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import optuna
import re
import warnings
import xgboost as xgb
import lightgbm as lgb
import category_encoders as ce
from catboost import CatBoostRegressor
from google.cloud import bigquery
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
from underthesea import text_normalize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('✅ All packages imported successfully')

## 3. Load data from BigQuery

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
GCP_KEY_PATH  = '/lakehouse/default/Files/real-estate-project-445516-c4ed0011894f.json'
BQ_TABLE      = 'real-estate-project-445516.real_estate_data.ads_data'
LAKEHOUSE_DIR = '/lakehouse/default/Files/training_data'
# ───────────────────────────────────────────────────────────

client = bigquery.Client.from_service_account_json(GCP_KEY_PATH)

query = f"""
    SELECT *
    FROM `{BQ_TABLE}`
    WHERE `Ngày gia hạn` >= '2026-03-01T00:00:00'
      AND `Ngày gia hạn` <= '2026-03-31T00:00:00'
"""

# Option 1: Load từ BigQuery (nếu có kết nối)
# df = client.query(query).to_dataframe()

# Option 2: Load từ file CSV (offline)
df = pd.read_csv(f'{LAKEHOUSE_DIR}/ads_data_2026_03.csv', low_memory=False)

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

## 4. Preprocessing, Deduplication & Advanced Outlier Removal

In [ ]:
# ── Filter ──────────────────────────────────────────────────────────
TOP_4_TYPES = ['căn hộ chung cư', 'nhà riêng', 'đất', 'nhà mặt phố']
# TOP_4_TYPES = ['căn hộ chung cư']

df = df[
    (df['Loại quảng cáo'] == 'Bán') &
    (df['Loại BĐS'].isin(TOP_4_TYPES)) &
    (df['Tỉnh thành phố'] == 'Hà Nội')
].copy()

for col in ['Khoảng giá', 'Diện tích', 'Tọa độ x', 'Tọa độ y']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['Khoảng giá'] > 1e8].dropna(subset=['Khoảng giá', 'Diện tích'])
print(f'After initial filter: {len(df):,} rows')

# ── Deduplication theo 15 cột cấu trúc ─────────────────────────────────
GROUP_COLS = [
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

df_dedup = (
    df.groupby(GROUP_COLS, dropna=False)
    .agg(
        Price=('Khoảng giá', lambda s: np.mean(np.unique(s[s.notna()])) if s.notna().any() else np.nan),
        Pháp_lý=('Pháp lý', 'first'),
        Nội_thất=('Nội thất', 'first'),
        Địa_chỉ_2=('Địa chỉ 2', 'first'),
        Tiêu_đề=('Tiêu đề', 'first'),
        Mô_tả=('Mô tả', 'first'),
        Phường=('Phường Xã Thị trấn', 'first'),
        Tọa_độ_x=('Tọa độ x', 'first'),
        Tọa_độ_y=('Tọa độ y', 'first'),
    )
    .reset_index()
    .rename(columns={
        'Pháp_lý': 'Pháp lý', 'Nội_thất': 'Nội thất', 'Địa_chỉ_2': 'Địa chỉ 2',
        'Tiêu_đề': 'Tiêu đề', 'Mô_tả': 'Mô tả', 'Phường': 'Phường Xã Thị trấn',
        'Tọa_độ_x': 'Tọa độ x', 'Tọa_độ_y': 'Tọa độ y',
    })
    .dropna(subset=['Price'])
)
print(f'After dedup: {len(df_dedup):,} rows')

# ── Isolation Forest Outlier Removal (thay vì drop top 1% đơn giản) ──────────
print('Applying Isolation Forest for intelligent outlier detection...')

df_if = df_dedup.copy()
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.02,  # ~2% outliers
    random_state=42,
    n_jobs=-1
)

outlier_mask_all = pd.Series(True, index=df_dedup.index)
for (loai, quan), g in df_dedup.groupby(['Loại BĐS', 'Quận']):
    if len(g) < 10:
        continue
    # Use log-price as features for Isolation Forest
    features_if = np.log1p(g[['Price']].values)
    preds = iso_forest.fit_predict(features_if)
    outlier_mask_all.loc[g.index] = (preds == 1)

df_final = df_dedup[outlier_mask_all].reset_index(drop=True)
print(f'After Isolation Forest: {len(df_final):,} rows')

display(df_final[['Price', 'Diện tích', 'Loại BĐS', 'Quận']].describe())

## 5. Feature Engineering (Nâng cao)

In [ ]:
METRO_STATIONS = [
    (21.028, 105.828), (21.015, 105.820), (21.015, 105.810),
    (21.030, 105.800), (21.002, 105.815), (20.975, 105.776),
]
CENTER_LAT, CENTER_LON = 21.0285, 105.8542

def extract_features(df):
    """
    Extract all features: structural, NLP (V11+V12+V13), spatial, and engineered.
    """
    print('Normalizing text and extracting features...')
    df = df.copy()
    df['clean_desc'] = df['Mô tả'].astype(str).apply(text_normalize).str.lower()
    desc = df['clean_desc']

    if 'Tiêu đề' in df.columns:
        df['clean_title'] = df['Tiêu đề'].astype(str).apply(text_normalize).str.lower()
        text = df['clean_title'] + ' ' + desc
    else:
        df['clean_title'] = ''
        text = desc

    # ── NLP features - V11 legacy ───────────────────────────────────────
    df['feat_oto']        = text.str.contains('xe hơi|ô tô|oto').astype(int)
    df['feat_tranh']      = text.str.contains('tránh').astype(int)
    df['feat_no_hau']     = text.str.contains('nở hậu').astype(int)
    df['feat_thang_may']  = text.str.contains('thang máy').astype(int)
    df['feat_kinh_doanh'] = text.str.contains('kinh doanh|buôn bán').astype(int)
    df['feat_mat_tien']   = text.str.contains('mặt phố|mặt đường').astype(int)
    df['feat_noi_that']   = text.str.contains('nội thất|đầy đủ|tiện nghi').astype(int)
    df['feat_so_do']      = text.str.contains('sổ đỏ|sổ hồng').astype(int)
    df['feat_chinh_chu']  = text.str.contains('chính chủ').astype(int)

    # ── NLP features - V11 new ──────────────────────────────────────
    df['feat_view_nui']      = text.str.contains('view núi|view đồi').astype(int)
    df['feat_view_ho_song']  = text.str.contains('view hồ|view sông|sát hồ|ven hồ').astype(int)
    df['feat_view_canh_dong']= text.str.contains('view cánh đồng|cánh đồng').astype(int)
    df['feat_khuon_vien']    = text.str.contains('sẵn ao|vườn cây|cây ăn trái|nhà vườn').astype(int)
    df['feat_nghi_duong']    = text.str.contains('nghỉ dưỡng|homestay|farmstay|villa').astype(int)
    df['feat_nha_xuong']     = text.str.contains('nhà xưởng').astype(int)
    df['feat_phan_lo']       = text.str.contains('phân lô').astype(int)
    df['feat_f0']            = text.str.contains('f0|chưa qua đầu tư').astype(int)
    df['feat_san_nha']       = text.str.contains('sẵn nhà|nhà cấp 4|ở ngay').astype(int)
    df['feat_duong_nhua']    = text.str.contains('đường nhựa').astype(int)
    df['feat_duong_betong']  = text.str.contains('đường bê tông').astype(int)
    df['feat_truc_chinh']    = text.str.contains('trục chính|đường tỉnh|tỉnh lộ|quốc lộ|đường lớn').astype(int)
    df['feat_phap_ly_chuan'] = text.str.contains('sẵn sổ|sang tên luôn|pháp lý chuẩn').astype(int)
    df['feat_du_lich']       = text.str.contains('khu du lịch|resort').astype(int)
    df['feat_truong_hoc']    = text.str.contains('trường học').astype(int)
    df['feat_cho']           = text.str.contains('chợ').astype(int)
    df['feat_nga_ba_tu']     = text.str.contains('ngã 3|ngã 4|ngã tư').astype(int)

    # ── NLP features - V12 ──────────────────────────────────────
    df['feat_view_bien']     = text.str.contains('biển').astype(int)
    df['feat_goc']           = text.str.contains('góc').astype(int)
    df['feat_cong_vien']     = text.str.contains('công viên').astype(int)
    df['feat_sieu_thi']      = text.str.contains('siêu thị|mart').astype(int)
    df['feat_benh_vien']     = text.str.contains('bệnh viện|bv').astype(int)
    df['feat_tttm']          = text.str.contains('trung tâm thương mại|tttm').astype(int)

    # ── House quality ────────────────────────────────────────
    df['feat_nha_moi']       = text.str.contains('mới xây|xây mới|nhà mới|mới tinh|mới hoàn thiện|bàn giao mới').astype(int)
    df['feat_cai_tao']       = text.str.contains('cải tạo|sửa chữa|nâng cấp|sơn sửa|tu sửa').astype(int)
    df['feat_nha_cu']        = text.str.contains('nhà cũ|cũ kỹ|xuống cấp|cần sửa|cũ nát').astype(int)

    # ── Multiple frontages ─────────────────────────────────
    df['feat_nhieu_mat_tien'] = text.str.contains('2 mặt tiền|3 mặt tiền|hai mặt tiền|mặt tiền trước sau|2 mặt phố|hai mặt phố|thông 2 ngả|thông 2 đường|thông ra 2 ngõ|thông 2 mặt phố').astype(int)
    df['feat_mat_tien_sau']   = text.str.contains('mặt tiền sau|mặt sau|mặt tiền phụ|ngõ thông').astype(int)

    # ── V13: Bổ sung NLP features ─────────────────────────
    df['feat_chung_cu_cao_cap'] = text.str.contains('chung cư cao cấp|chung cư mini|penthouse|duplex|sky villa').astype(int)
    df['feat_ho_boi']           = text.str.contains('hồ bơi|bể bơi').astype(int)
    df['feat_gym']              = text.str.contains('gym|fitness|tập gym|phòng tập').astype(int)
    df['feat_ban_cong']         = text.str.contains('ban công|loggia|sân thượng').astype(int)
    df['feat_an_ninh']          = text.str.contains('an ninh|bảo vệ|bảo vệ 24/24|an ninh tốt|camera').astype(int)
    df['feat_de_xe']            = text.str.contains('chỗ để xe|hầm để xe|tầng hầm').astype(int)
    df['feat_moi_gioi']         = text.str.contains('môi giới|call|liên hệ|hotline').astype(int)
    df['feat_gia_tot']          = text.str.contains('giá tốt|giá rẻ|giá mềm|giá đầu tư|rẻ nhất').astype(int)
    df['feat_thuong_luong']     = text.str.contains('thương lượng|có thương lượng|còn thương lượng').astype(int)

    # ── Geographic distance ───────────────────────────────
    df['dist_to_metro'] = df.apply(
        lambda r: min(np.sqrt((r['Tọa độ x'] - mx)**2 + (r['Tọa độ y'] - my)**2)
                      for mx, my in METRO_STATIONS), axis=1
    )
    df['dist_to_center'] = np.sqrt(
        (df['Tọa độ x'] - CENTER_LAT)**2 + (df['Tọa độ y'] - CENTER_LON)**2
    )

    # ── Combined key ────────────────────────────────────
    df['type_dist'] = df['Loại BĐS'].astype(str) + '_' + df['Quận'].astype(str)

    # ── Clean numeric columns ──────────────────────────────
    all_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào']
    int_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh']

    for col in all_cols:
        converted = pd.to_numeric(
            df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce'
        )
        if col in int_cols:
            df[col] = converted.astype(float)  # float thay vi Int64 de tranh pd.NA (CatBoost khong support)
        else:
            df[col] = converted

    df['can_goc'] = df['Căn góc'].astype(str).str.lower().isin(
        ['có', 'yes', '1', 'true', 'căn góc']
    ).astype(int)

    # ── Engineered features ──────────────────────────────
    df['dien_tich_per_tang'] = df['Diện tích'] / df['Số tầng'].replace(0, np.nan).fillna(1)
    df['mat_tien_x_tang']    = df['Mặt tiền'] * df['Số tầng']
    df['log_dien_tich']      = np.log1p(df['Diện tích'])
    df['vuong_area']         = np.sqrt(df['Diện tích'])
    # price_per_m2 KHONG duoc dua vao features vi bi data leakage (duoc tinh truc tiep tu Price - target)
    # Chi dung price_per_m2 de tinh ward-level statistics sau khi split
    df['price_per_m2']       = df['Price'] / df['Diện tích']

    return df

df_final = extract_features(df_final)
print(f'✅ Feature extraction done. Shape: {df_final.shape}')

## 5b. TF-IDF + SVD: Deep Text Features từ Mô tả

In [ ]:
print('Building TF-IDF + SVD text features...')

full_text = (df_final['clean_title'] + ' ' + df_final['clean_desc']).fillna('')

tfidf = TfidfVectorizer(
    max_features=2000,
    stop_words=None,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)
tfidf_matrix = tfidf.fit_transform(full_text)
print(f'  TF-IDF matrix shape: {tfidf_matrix.shape}')

N_SVD_COMPONENTS = 25
svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=42)
svd_features = svd.fit_transform(tfidf_matrix)
print(f'  SVD explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}')

for i in range(N_SVD_COMPONENTS):
    df_final[f'desc_svd_{i}'] = svd_features[:, i]

print('✅ TF-IDF + SVD text features extracted')

In [ ]:
df_final.info()

## 6. Spatial Clustering & Train/Test Split

In [ ]:
NLP_FEATURES_V11 = [
    'feat_oto', 'feat_tranh', 'feat_no_hau', 'feat_thang_may', 'feat_kinh_doanh',
    'feat_mat_tien', 'feat_noi_that', 'feat_so_do', 'feat_chinh_chu',
    'feat_view_nui', 'feat_view_ho_song', 'feat_view_canh_dong', 'feat_khuon_vien',
    'feat_nghi_duong', 'feat_nha_xuong', 'feat_phan_lo', 'feat_f0', 'feat_san_nha',
    'feat_duong_nhua', 'feat_duong_betong', 'feat_truc_chinh', 'feat_phap_ly_chuan',
    'feat_du_lich', 'feat_truong_hoc', 'feat_cho', 'feat_nga_ba_tu',
    'feat_view_bien', 'feat_goc', 'feat_cong_vien', 'feat_sieu_thi',
    'feat_benh_vien', 'feat_tttm',
    'feat_nha_moi', 'feat_cai_tao', 'feat_nha_cu',
    'feat_nhieu_mat_tien', 'feat_mat_tien_sau',
]

NLP_FEATURES_V13 = [
    'feat_chung_cu_cao_cap', 'feat_ho_boi', 'feat_gym', 'feat_ban_cong',
    'feat_an_ninh', 'feat_de_xe', 'feat_moi_gioi', 'feat_gia_tot', 'feat_thuong_luong',
]

SVD_FEATURES = [f'desc_svd_{i}' for i in range(N_SVD_COMPONENTS)]

STRUCTURAL_FEATURES = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1', 'Diện tích',
    'Tọa độ x', 'Tọa độ y', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh',
    'Mặt tiền', 'Đường vào', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'can_goc',
]

ENGINEERED_FEATURES = [
    'dist_to_metro', 'dist_to_center', 'type_dist', 'loc_cluster',
    'dien_tich_per_tang', 'mat_tien_x_tang',
    'log_dien_tich', 'vuong_area',
]

CATEGORICAL = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1',
    'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'type_dist', 'loc_cluster'
]

print('Creating spatial clusters...')
coords = df_final[['Tọa độ x', 'Tọa độ y']].copy()
coords['Tọa độ x'] = coords['Tọa độ x'].fillna(coords['Tọa độ x'].median())
coords['Tọa độ y'] = coords['Tọa độ y'].fillna(coords['Tọa độ y'].median())

kmeans = MiniBatchKMeans(n_clusters=400, random_state=42, n_init=3)
df_final['loc_cluster'] = kmeans.fit_predict(coords)

ALL_FEATURES = (
    STRUCTURAL_FEATURES +
    NLP_FEATURES_V11 +
    NLP_FEATURES_V13 +
    SVD_FEATURES +
    ENGINEERED_FEATURES
)

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  - Structural: {len(STRUCTURAL_FEATURES)}')
print(f'  - NLP V11: {len(NLP_FEATURES_V11)}')
print(f'  - NLP V13: {len(NLP_FEATURES_V13)}')
print(f'  - SVD text: {len(SVD_FEATURES)}')
print(f'  - Engineered: {len(ENGINEERED_FEATURES)}')

In [ ]:
# ── Prepare X, y ────────────────────────────────────────────
X = df_final[ALL_FEATURES].copy()
y = np.log1p(df_final['Price'])
groups = df_final['loc_cluster'].values

print(f'Full dataset: {X.shape}')

# ── Train/Test split ──────────────────────────────────────
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.15, random_state=42,
)

# ── Ward-level statistics (chỉ từ TRAIN để tránh leakage) ──────────
# Tính median PPM (Price Per M2) theo từng phường
train_ward_vals = X_train['Địa chỉ 1'].copy()
train_ward_ppm = df_final.loc[X_train.index, 'price_per_m2']
ward_ppm_median = train_ward_ppm.groupby(train_ward_vals.values).transform('median')
X_train['ward_ppm_med'] = ward_ppm_median.fillna(train_ward_ppm.median())

# Áp dụng cho test: map từ phường → median từ train
ward_ppm_map = df_final.loc[X_train.index].groupby('Địa chỉ 1')['price_per_m2'].median().to_dict()
X_test['ward_ppm_med'] = X_test['Địa chỉ 1'].map(ward_ppm_map).fillna(train_ward_ppm.median())

ALL_FEATURES = ALL_FEATURES + ['ward_ppm_med']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Total features with ward stats: {len(ALL_FEATURES)}')

In [ ]:
encoder = ce.TargetEncoder(cols=CATEGORICAL)
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc  = encoder.transform(X_test)

print(f'Train encoded: {X_train_enc.shape}')
print(f'Test encoded: {X_test_enc.shape}')

## 7. Optuna Tuning — XGBoost (Nâng cao)

In [ ]:
N_TRIALS = 30

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.03, log=True),
        'max_depth':         trial.suggest_int('max_depth', 8, 16),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 15),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 10.0),
        'tree_method': 'hist',
        'verbosity': 0,
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    for _, (train_idx, val_idx) in enumerate(kf.split(X_train_enc, y_train, groups_train)):
        X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = xgb.XGBRegressor(**params, early_stopping_rounds=50, eval_metric='mape')
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return np.mean(scores)

print(f'Starting XGBoost tuning with {N_TRIALS} trials (3-fold CV each)...')
study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGB best CV MAPE: {study_xgb.best_value*100:.2f}%')
print(f'Best params: {study_xgb.best_params}')

## 8. Optuna Tuning — LightGBM (Nâng cao)

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.03, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 255, 1023),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 10.0),
        'min_split_gain':   trial.suggest_float('min_split_gain', 0.0, 1.0),
        'max_bin':          trial.suggest_int('max_bin', 255, 512),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'device': 'cpu',
        'verbose': -1,
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train_enc, y_train, groups_train):
        X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mape',
              callbacks=[lgb.log_evaluation(0)])
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return np.mean(scores)

print(f'Starting LightGBM tuning with {N_TRIALS} trials (3-fold CV each)...')
study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LGB best CV MAPE: {study_lgb.best_value*100:.2f}%')
print(f'Best params: {study_lgb.best_params}')

## 8b. Optuna Tuning — CatBoost

In [ ]:
print('Starting CatBoost tuning...')

def objective_cat(trial):
    params = {
        'iterations':       trial.suggest_int('iterations', 1000, 3000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth':            trial.suggest_int('depth', 6, 12),
        'l2_leaf_reg':      trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count':     trial.suggest_int('border_count', 128, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':  trial.suggest_float('random_strength', 0.0, 1.0),
        'verbose': 0,
        'loss_function': 'RMSE',
        'eval_metric': 'MAPE',
        'od_type': 'Iter',
        'od_wait': 50,
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train_enc, y_train, groups_train):
        X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = CatBoostRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return np.mean(scores)

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=min(N_TRIALS, 15), show_progress_bar=True)

print(f'\n✅ CatBoost best CV MAPE: {study_cat.best_value*100:.2f}%')
print(f'Best params: {study_cat.best_params}')

## 9. Final Training & Advanced Ensemble (Stacking)

In [ ]:
print('=' * 60)
print('  V13: Training Final Models + Stacking Ensemble')
print('=' * 60)

# ── 1. Train XGBoost ──────────────────────────────────────
print('\n[1/3] Training final XGBoost...')
m_xgb = xgb.XGBRegressor(**study_xgb.best_params, verbosity=0)
m_xgb.fit(X_train_enc, y_train)
p_xgb = np.expm1(m_xgb.predict(X_test_enc))

# ── 2. Train LightGBM ──────────────────────────────────
print('[2/3] Training final LightGBM...')
m_lgb = lgb.LGBMRegressor(**study_lgb.best_params, verbose=-1)
m_lgb.fit(X_train_enc, y_train)
p_lgb = np.expm1(m_lgb.predict(X_test_enc))

# ── 3. Train CatBoost ─────────────────────────────────
print('[3/3] Training final CatBoost...')
m_cat = CatBoostRegressor(**study_cat.best_params, verbose=0)
m_cat.fit(X_train_enc, y_train, verbose=0)
p_cat = np.expm1(m_cat.predict(X_test_enc))

y_true = np.expm1(y_test)

# ── Method 1: Grid search weighted blend (3 models) ─────────────
print('\nOptimizing ensemble weights (grid search)...')
best_mape_blend = 999
best_w_xgb, best_w_lgb, best_w_cat = 0.34, 0.33, 0.33
for w1 in np.arange(0.0, 1.01, 0.1):
    for w2 in np.arange(0.0, 1.01 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 < 0:
            continue
        blended = w1 * p_xgb + w2 * p_lgb + w3 * p_cat
        mape = mean_absolute_percentage_error(y_true, blended)
        if mape < best_mape_blend:
            best_mape_blend = mape
            best_w_xgb, best_w_lgb, best_w_cat = w1, w2, w3

y_pred_blend = best_w_xgb * p_xgb + best_w_lgb * p_lgb + best_w_cat * p_cat
print(f'  Best blend weights: XGB={best_w_xgb:.2f}, LGB={best_w_lgb:.2f}, Cat={best_w_cat:.2f}')
print(f'  Blend MAPE: {best_mape_blend*100:.2f}%')

# ── Method 2: Stacking with Ridge meta-model ────────────────
print('\nTraining Stacking Ensemble (Ridge meta-model)...')

kf = GroupKFold(n_splits=5)
oof_xgb = np.zeros(len(X_train_enc))
oof_lgb = np.zeros(len(X_train_enc))
oof_cat = np.zeros(len(X_train_enc))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_enc, y_train, groups_train)):
    X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb_fold = xgb.XGBRegressor(**study_xgb.best_params, verbosity=0)
    xgb_fold.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_fold.predict(X_val)
    
    lgb_fold = lgb.LGBMRegressor(**study_lgb.best_params, verbose=-1)
    lgb_fold.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb_fold.predict(X_val)
    
    cat_fold = CatBoostRegressor(**study_cat.best_params, verbose=0)
    cat_fold.fit(X_tr, y_tr, verbose=0)
    oof_cat[val_idx] = cat_fold.predict(X_val)
    print(f'  Fold {fold+1}/5 complete')

meta_features_train = np.column_stack([oof_xgb, oof_lgb, oof_cat])
meta_features_test = np.column_stack([p_xgb, p_lgb, p_cat])

meta_model = Ridge(alpha=0.5)
meta_model.fit(meta_features_train, y_train.values)
print(f'  Meta-model coefficients: XGB={meta_model.coef_[0]:.3f}, LGB={meta_model.coef_[1]:.3f}, Cat={meta_model.coef_[2]:.3f}')

meta_pred_log = meta_model.predict(meta_features_test)
meta_pred_log = np.clip(meta_pred_log, -50, 50)
y_pred_stack = np.expm1(meta_pred_log)

# Remove any NaN/inf from y_true or predictions before MAPE
stack_valid = np.isfinite(y_pred_stack) & np.isfinite(y_true)
if not stack_valid.all():
    print(f'  ⚠ Filtered out {(~stack_valid).sum()} invalid samples before MAPE')
y_true_stack = y_true[stack_valid]
y_pred_stack = y_pred_stack[stack_valid]
mape_stack = mean_absolute_percentage_error(y_true_stack, y_pred_stack)
print(f'  Stacking MAPE: {mape_stack*100:.2f}%')

# ── Chọn phương pháp tốt nhất ──────────────────────────
if mape_stack < best_mape_blend:
    print(f'\n*** Using STACKING ensemble (MAPE: {mape_stack*100:.2f}%)')
    y_pred_final = y_pred_stack
    best_mape = mape_stack
    ensemble_method = 'stacking'
else:
    print(f'\n*** Using BLEND ensemble (MAPE: {best_mape_blend*100:.2f}%)')
    y_pred_final = y_pred_blend
    best_mape = best_mape_blend
    ensemble_method = 'blend'

r2 = r2_score(np.log1p(y_true), np.log1p(y_pred_final))
print(f'   R² score: {r2:.4f}')

eval_df = X_test[['Loại BĐS']].copy()
eval_df['y_true'] = y_true.values
eval_df['y_pred'] = y_pred_final
print('\nMAPE by BĐS type:')
mape_by_type = (
    eval_df.groupby('Loại BĐS')
    .apply(lambda x: mean_absolute_percentage_error(x['y_true'], x['y_pred']) * 100)
    .rename('MAPE (%)')
    .round(2)
)
print(mape_by_type.to_string() if not mape_by_type.empty else '  (single type only)')

## 9b. SHAP Feature Selection (Tự động loại bỏ features không quan trọng)

In [ ]:
try:
    import shap
    print('Calculating SHAP feature importance... (may take a few minutes)')
    
    sample_size = min(500, len(X_test_enc))
    X_sample = X_test_enc.iloc[:sample_size]
    
    explainer_xgb = shap.TreeExplainer(m_xgb)
    shap_values_xgb = explainer_xgb.shap_values(X_sample)
    
    shap_mean = np.abs(shap_values_xgb).mean(axis=0)
    fi_shap = pd.Series(shap_mean, index=ALL_FEATURES).sort_values(ascending=False)
    
    print(f'\nTop 20 most important features (by SHAP):')
    print(fi_shap.head(20).to_string())
    
    threshold = fi_shap.quantile(0.05)
    low_importance = fi_shap[fi_shap < threshold].index.tolist()
    if low_importance:
        print(f'\nLow importance features (bottom 5%%, could be removed):')
        print(f'  {low_importance}')
    
    print('\n✅ SHAP analysis complete')
except ImportError:
    print('SHAP not installed, skipping feature importance analysis.')
except Exception as e:
    print(f'SHAP analysis skipped: {e}')

## 10. Feature Importance Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

fi_xgb = pd.Series(m_xgb.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True).tail(30)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Plot 1: XGBoost importance
fi_xgb.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('XGBoost — Top 30 Feature Importance (V13)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Plot 2: Error distribution
error_pct = (y_pred_final - y_true) / y_true * 100
axes[1].hist(error_pct, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title(f'Prediction Error Distribution (MAPE={best_mape*100:.2f}%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Error (%)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{LAKEHOUSE_DIR}/feature_importance_v13.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n*** Error Statistics:')
print(f'  Mean error: {error_pct.mean():.2f}%')
print(f'  Median error: {error_pct.median():.2f}%')
print(f'  Std error: {error_pct.std():.2f}%')
print(f'  Within \u00b110%: {(np.abs(error_pct) <= 10).mean()*100:.1f}%')
print(f'  Within \u00b120%: {(np.abs(error_pct) <= 20).mean()*100:.1f}%')
print(f'  Within \u00b130%: {(np.abs(error_pct) <= 30).mean()*100:.1f}%')

## 11. Save Models & Metadata to Lakehouse

In [ ]:
joblib.dump(m_xgb,   f'{LAKEHOUSE_DIR}/master_xgb_v13.joblib')
joblib.dump(m_lgb,   f'{LAKEHOUSE_DIR}/master_lgb_v13.joblib')
joblib.dump(m_cat,   f'{LAKEHOUSE_DIR}/master_cat_v13.joblib')
joblib.dump(encoder, f'{LAKEHOUSE_DIR}/master_encoder_v13.joblib')
joblib.dump(kmeans,  f'{LAKEHOUSE_DIR}/master_kmeans_v13.joblib')
joblib.dump(meta_model, f'{LAKEHOUSE_DIR}/master_meta_v13.joblib')
joblib.dump(tfidf,   f'{LAKEHOUSE_DIR}/master_tfidf_v13.joblib')
joblib.dump(svd,     f'{LAKEHOUSE_DIR}/master_svd_v13.joblib')

meta = {
    'version': 'V13',
    'mape': f'{best_mape*100:.2f}%',
    'r2_score': round(float(r2), 4),
    'ensemble_method': ensemble_method,
    'xgb_weight': best_w_xgb if ensemble_method == 'blend' else float(meta_model.coef_[0]),
    'lgb_weight': best_w_lgb if ensemble_method == 'blend' else float(meta_model.coef_[1]),
    'cat_weight': best_w_cat if ensemble_method == 'blend' else float(meta_model.coef_[2]),
    'features': ALL_FEATURES,
    'n_features': len(ALL_FEATURES),
    'categorical': CATEGORICAL,
    'xgb_params': study_xgb.best_params,
    'lgb_params': study_lgb.best_params,
    'cat_params': study_cat.best_params,
    'n_train': len(X_train),
    'n_test': len(X_test),
    'svd_components': N_SVD_COMPONENTS,
    'improvements': [
        'CatBoost added to triple ensemble',
        'Stacking with Ridge meta-model',
        'TF-IDF + SVD deep text features (25 components)',
        'GroupKFold cross-validation by location cluster',
        'Expanded Optuna hyperparameters (gamma, reg_alpha, reg_lambda, etc.)',
        'Isolation Forest for intelligent outlier detection',
        'Ward-level PPM median statistics',
        'SHAP feature importance analysis',
        'Additional V13 NLP features',
    ],
}

with open(f'{LAKEHOUSE_DIR}/model_meta_v13.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('✅ V13 Models saved to Lakehouse:')
print('   ├── master_xgb_v13.joblib')
print('   ├── master_lgb_v13.joblib')
print('   ├── master_cat_v13.joblib')
print('   ├── master_encoder_v13.joblib')
print('   ├── master_kmeans_v13.joblib')
print('   ├── master_meta_v13.joblib')
print('   ├── master_tfidf_v13.joblib')
print('   ├── master_svd_v13.joblib')
print('   └── model_meta_v13.json')
print(f'\n*** FINAL V13 MAPE: {best_mape*100:.2f}%')